# 🍷 Sommo AI v1 — Wine Expert LLM

**A fine-tuned language model for wine recommendations, food pairings, and sommelier-level advice.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanarkan/sommo-ai/blob/main/Sommo_AI_v1.ipynb)
[![Model on HF](https://img.shields.io/badge/🤗-Model-yellow)](https://huggingface.co/gokhanarkan/sommo-7b-v1)
[![App Store](https://img.shields.io/badge/📱-sommo.app-black)](https://sommo.app)

---

## About This Project

**Sommo AI** is an open-source wine expert model built on Qwen 2.5-7B-Instruct using LoRA fine-tuning. This notebook documents the complete training pipeline from data collection to deployment.

### What This Model Can Do

- 🍽️ **Food Pairing** — Recommend wines for specific dishes with reasoning
- 🍇 **Wine Knowledge** — Explain grape varieties, regions, and winemaking
- 💰 **Recommendations** — Suggest wines by budget, occasion, or preference
- 📝 **Tasting Notes** — Describe wines with professional vocabulary

### Version History

| Version | Description | Status |
|---------|-------------|--------|
| **v1** | This notebook — Proof of concept with 7 datasets | Public |
| **v2** | Production model powering [sommo.app](https://sommo.app) | Private |

> **Note:** The [Sommo iOS app](https://sommo.app) uses an enhanced v2 model with additional proprietary training data. This v1 release demonstrates the methodology and serves as a starting point for the community.

---

## Training Overview

| Component | Choice | Rationale |
|-----------|--------|----------|
| **Base Model** | Qwen 2.5-7B-Instruct | Best quality/speed tradeoff for domain-specific tasks |
| **Method** | LoRA (r=64) | Memory-efficient fine-tuning without full weight updates |
| **Hardware** | H100 80GB | Fast training (~3-4 hours), BF16 precision |
| **Data** | ~100K conversations | 7 curated sources + synthetic generation |

---

## Data Sources

We curated 7 complementary datasets covering different aspects of wine knowledge:

| # | Dataset | Records | Purpose |
|---|---------|---------|--------|
| 1 | WineEnthusiast Reviews | 130K | Professional tasting vocabulary |
| 2 | Alfredodeza Wine Ratings | 33K | Detailed review structure |
| 3 | X-Wines | 1K+ | Wine metadata and food pairings |
| 4 | Vivino Rating and Price | 13.8K | Consumer perspective and pricing |
| 5 | Wine Food Pairing NLP | ~10K | Pairing logic and descriptors |
| 6 | Wikipedia Wine Articles | 50+ | Factual knowledge base |
| 7 | Synthetic QA (Gemini) | 45 | High-quality conversation examples |

The synthetic data (Dataset 7) is crucial — it teaches the model *how to respond* as a sommelier, not just *what to say*.

---

# Part 1: Environment Setup

We start by configuring the Colab environment with the correct package versions. The key constraint is pinning `datasets==4.3.0` for Unsloth compatibility.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


In [ ]:
!pip install -q datasets==4.3.0
!pip install -q unsloth
!pip install -q --upgrade transformers trl peft accelerate bitsandbytes
!pip install -q sentencepiece wikipedia-api
!pip install -q kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 54.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.1/381.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 147.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import datasets
assert datasets.__version__.startswith("4.3"), f"Expected datasets 4.3.x, got {datasets.__version__}"
print(f"datasets {datasets.__version__}")

datasets 4.3.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/sommo_ai"
for subdir in ["checkpoints", "data/raw", "data/processed", "models"]:
    os.makedirs(f"{PROJECT_DIR}/{subdir}", exist_ok=True)

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")

Mounted at /content/drive
Project directory: /content/drive/MyDrive/sommo_ai


### API Configuration

We use Colab Secrets for secure credential management. Add these in the key sidebar:

- `KAGGLE_USERNAME` — Your Kaggle username
- `KAGGLE_KEY` — From kaggle.com/account then Create New API Token

Gemini AI is available natively in Colab Pro+ without additional configuration.

In [ ]:
from google.colab import userdata
import json

try:
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
    KAGGLE_AVAILABLE = bool(kaggle_user and kaggle_key)

    if KAGGLE_AVAILABLE:
        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(f"{kaggle_dir}/kaggle.json", "w") as f:
            json.dump({"username": kaggle_user, "key": kaggle_key}, f)
        os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)
        print("Kaggle configured")
except Exception:
    KAGGLE_AVAILABLE = False
    print("Kaggle not configured")

try:
    from google.colab import ai
    GEMINI_AVAILABLE = True
    print("Gemini AI available")
except ImportError:
    GEMINI_AVAILABLE = False
    print("Gemini AI not available")

Kaggle configured
Gemini AI available


---

# Part 2: Data Collection

Each dataset contributes something unique:

- **WineEnthusiast** provides professional critic vocabulary
- **Alfredodeza** offers longer, more detailed reviews
- **X-Wines** includes food pairing data (the Harmonize field)
- **Vivino** adds consumer pricing context
- **Wikipedia** grounds the model in factual knowledge

In [ ]:
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import random

all_datasets = {}

In [ ]:
print("Loading WineEnthusiast...")

try:
    wine_enthusiast = load_dataset("spawn99/wine-reviews", split="train")
    all_datasets['wine_enthusiast'] = wine_enthusiast.to_pandas()
    print(f"WineEnthusiast: {len(all_datasets['wine_enthusiast']):,} reviews")
except Exception as e:
    print(f"WineEnthusiast failed: {e}")

Loading WineEnthusiast...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/39.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/5.69M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/196630 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/28090 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/56181 [00:00<?, ? examples/s]

WineEnthusiast: 196,630 reviews


In [ ]:
print("Loading Alfredodeza...")

try:
    alfredo = load_dataset("alfredodeza/wine-ratings", split="train")
    all_datasets['alfredo'] = alfredo.to_pandas()
    print(f"Alfredodeza: {len(all_datasets['alfredo']):,} reviews")
except Exception as e:
    print(f"Alfredodeza failed: {e}")

Loading Alfredodeza...


README.md:   0%|          | 0.00/502 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/32780 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Alfredodeza: 32,780 reviews


In [ ]:
print("Loading X-Wines...")

if KAGGLE_AVAILABLE:
    try:
        !kaggle datasets download -d rogerioxavier/x-wines-slim-version -p data/raw/ --unzip -q

        import glob
        csv_files = glob.glob("data/raw/*wines*.csv") + glob.glob("data/raw/*Wines*.csv")

        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                if len(df) > 0:
                    all_datasets['xwines'] = df
                    print(f"X-Wines: {len(df):,} wines")
                    break
            except Exception:
                continue
    except Exception as e:
        print(f"X-Wines failed: {e}")
else:
    print("X-Wines skipped (no Kaggle credentials)")

Loading X-Wines...
Dataset URL: https://www.kaggle.com/datasets/rogerioxavier/x-wines-slim-version
License(s): DbCL-1.0
X-Wines: 1,007 wines


In [ ]:
print("Loading Vivino...")

if KAGGLE_AVAILABLE:
    try:
        !kaggle datasets download -d budnyak/wine-rating-and-price -p data/raw/vivino --unzip -q

        vivino_dfs = []
        for wine_type in ['Red', 'White', 'Rose', 'Sparkling']:
            path = f"data/raw/vivino/{wine_type}.csv"
            if os.path.exists(path):
                df = pd.read_csv(path)
                df['wine_type'] = wine_type
                vivino_dfs.append(df)

        if vivino_dfs:
            all_datasets['vivino'] = pd.concat(vivino_dfs, ignore_index=True)
            print(f"Vivino: {len(all_datasets['vivino']):,} wines")
    except Exception as e:
        print(f"Vivino failed: {e}")
else:
    print("Vivino skipped (no Kaggle credentials)")

Loading Vivino...
Dataset URL: https://www.kaggle.com/datasets/budnyak/wine-rating-and-price
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
Vivino: 13,834 wines


In [ ]:
print("Loading Wine Food Pairing...")

try:
    !rm -rf data/raw/wine_food_pairing 2>/dev/null
    !git clone --depth 1 https://github.com/RoaldSchuring/wine_food_pairing.git data/raw/wine_food_pairing 2>&1 | tail -1

    import glob
    pairing_data = {}

    for base in ["data/raw/wine_food_pairing/data", "data/raw/wine_food_pairing"]:
        for csv_file in glob.glob(f"{base}/*.csv"):
            try:
                df = pd.read_csv(csv_file)
                name = os.path.basename(csv_file).replace('.csv', '')
                pairing_data[name] = df
            except Exception:
                continue

    if pairing_data:
        all_datasets['food_pairing'] = pairing_data
        total = sum(len(df) for df in pairing_data.values())
        print(f"Wine Food Pairing: {total:,} records")
    else:
        print("Wine Food Pairing: no CSV files found")
except Exception as e:
    print(f"Wine Food Pairing failed: {e}")

Loading Wine Food Pairing...
Cloning into 'data/raw/wine_food_pairing'...
Wine Food Pairing: 6,818 records


In [ ]:
print("Loading Wikipedia...")

import wikipediaapi

WINE_TOPICS = [
    "Bordeaux_wine", "Burgundy_wine", "Champagne_(wine_region)", "Napa_Valley_AVA",
    "Tuscany_wine", "Rioja_(wine)", "Barossa_Valley", "Mosel_(wine_region)",
    "Willamette_Valley_AVA", "Mendoza_wine", "Marlborough_(wine_region)",
    "Cabernet_Sauvignon", "Merlot", "Pinot_noir", "Chardonnay", "Sauvignon_blanc",
    "Riesling", "Syrah", "Zinfandel", "Malbec", "Tempranillo", "Sangiovese",
    "Pinot_grigio", "Nebbiolo", "Grenache",
    "Wine_tasting", "Terroir", "Winemaking", "Malolactic_fermentation",
    "Oak_(wine)", "Wine_and_food_pairing", "Decanting", "Wine_cellar",
    "Tannin_(wine)", "Sommelier", "Natural_wine", "Organic_wine", "Orange_wine",
    "Sparkling_wine", "Rose", "Dessert_wine", "Fortified_wine",
    "Champagne", "Prosecco", "Port_wine", "Sherry", "Madeira_wine",
]

try:
    wiki = wikipediaapi.Wikipedia(user_agent='SommoAI/1.0', language='en')
    articles = []

    for topic in tqdm(WINE_TOPICS, desc="Fetching"):
        page = wiki.page(topic)
        if page.exists():
            articles.append({
                'title': page.title,
                'summary': page.summary,
                'text': page.text[:8000]
            })

    all_datasets['wikipedia'] = pd.DataFrame(articles)
    print(f"Wikipedia: {len(articles)} articles")
except Exception as e:
    print(f"Wikipedia failed: {e}")

Loading Wikipedia...


Fetching: 100%|██████████| 47/47 [00:18<00:00,  2.54it/s]

Wikipedia: 46 articles


In [ ]:
print("\n" + "="*50)
print("DATA COLLECTION SUMMARY")
print("="*50)

total = 0
for name, data in all_datasets.items():
    if isinstance(data, pd.DataFrame):
        count = len(data)
    elif isinstance(data, dict):
        count = sum(len(df) for df in data.values() if isinstance(df, pd.DataFrame))
    else:
        count = 0
    total += count
    print(f"  {name}: {count:,}")

print(f"\n  Total: {total:,} records")


DATA COLLECTION SUMMARY
  wine_enthusiast: 196,630
  alfredo: 32,780
  xwines: 1,007
  vivino: 13,834
  food_pairing: 6,818
  wikipedia: 46

  Total: 251,115 records


---

# Part 3: Data Processing

We transform raw data into conversation format. The key insight is that we are not just teaching facts — we are teaching *how to respond* as a sommelier.

In [ ]:
SOMMO_SYSTEM = """You are Sommo, an expert sommelier with decades of experience in wine selection, food pairing, and wine education. You have extensive knowledge of wine regions worldwide, grape varieties and their characteristics, winemaking techniques, and food pairing principles. You communicate in a warm, knowledgeable manner - approachable for beginners yet sophisticated enough for experts. You use vivid, sensory language when describing wines and always consider the person's context, budget, and preferences."""

def create_conversation(user_msg, assistant_msg):
    return {
        "conversations": [
            {"role": "system", "content": SOMMO_SYSTEM},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg}
        ]
    }

In [ ]:
def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    return ' '.join(text.split()).strip()

def is_valid(text, min_len=50, max_len=2000):
    return min_len <= len(text) <= max_len

In [ ]:
training_data = []

WINE_QUESTIONS = [
    "Tell me about {wine}.",
    "What can you tell me about this {variety} from {region}?",
    "Describe this {variety} wine.",
    "I'm considering {wine}. What's it like?",
]

In [ ]:
if 'wine_enthusiast' in all_datasets:
    print("Processing WineEnthusiast...")
    df = all_datasets['wine_enthusiast']
    count = 0

    for _, row in tqdm(df.iterrows(), total=len(df)):
        description = clean_text(row.get('description', ''))
        if not is_valid(description):
            continue

        wine = row.get('title', 'this wine')
        variety = row.get('variety', 'wine')
        region = row.get('region_1', row.get('province', ''))
        country = row.get('country', '')
        points = row.get('points', '')
        price = row.get('price', '')

        parts = []
        if variety and region and country:
            parts.append(f"This {variety} from {region}, {country} is a noteworthy selection.")
        parts.append(description)
        if points:
            parts.append(f"It earned {points} points from critics.")
        if price and str(price) not in ['', 'nan', 'None']:
            try:
                parts.append(f"At around ${int(float(price))}, it represents solid value.")
            except ValueError:
                pass

        question = random.choice(WINE_QUESTIONS).format(
            wine=wine, variety=variety or 'wine', region=region or 'the region'
        )
        training_data.append(create_conversation(question, " ".join(parts)))
        count += 1

    print(f"  {count:,} conversations")

Processing WineEnthusiast...


100%|██████████| 196630/196630 [00:14<00:00, 13684.45it/s]

  196,590 conversations


In [ ]:
if 'alfredo' in all_datasets:
    print("Processing Alfredodeza...")
    df = all_datasets['alfredo']
    count = 0

    for _, row in tqdm(df.iterrows(), total=len(df)):
        notes = clean_text(row.get('notes', ''))
        if not is_valid(notes):
            continue

        name = row.get('name', row.get('wine', 'this wine'))
        rating = row.get('rating', '')

        response = notes
        if rating:
            response += f" This wine has a rating of {rating}."

        question = f"Tell me about {name}."
        training_data.append(create_conversation(question, response))
        count += 1

    print(f"  {count:,} conversations")

Processing Alfredodeza...


100%|██████████| 32780/32780 [00:02<00:00, 13969.38it/s]

  31,274 conversations


In [ ]:
if 'xwines' in all_datasets:
    print("Processing X-Wines...")
    df = all_datasets['xwines']
    count = 0

    cols = {c.lower(): c for c in df.columns}

    for _, row in tqdm(df.iterrows(), total=len(df)):
        wine_name = row.get(cols.get('winename', 'WineName'),
                           row.get(cols.get('name', 'Name'), 'this wine'))
        wine_type = row.get(cols.get('type', 'Type'), '')
        grapes = row.get(cols.get('grapes', 'Grapes'), '')
        country = row.get(cols.get('country', 'Country'), '')
        region = row.get(cols.get('regionname', 'RegionName'), '')
        harmonize = row.get(cols.get('harmonize', 'Harmonize'), '')

        if wine_name and (grapes or country):
            parts = []
            if wine_type and grapes:
                parts.append(f"{wine_name} is a {str(wine_type).lower()} wine made from {grapes}.")
            if region and country:
                parts.append(f"It comes from {region}, {country}.")

            if parts:
                training_data.append(create_conversation(
                    f"Tell me about {wine_name}.", " ".join(parts)
                ))
                count += 1

        if harmonize and str(harmonize) not in ['', 'nan', 'None', '[]']:
            response = f"{wine_name} pairs wonderfully with {harmonize}."
            if grapes:
                response += f" The characteristics of {grapes} complement these flavors."

            training_data.append(create_conversation(
                f"What food goes well with {wine_name}?", response
            ))
            count += 1

    print(f"  {count:,} conversations")

Processing X-Wines...


100%|██████████| 1007/1007 [00:00<00:00, 14720.29it/s]

  2,014 conversations


In [ ]:
if 'vivino' in all_datasets:
    print("Processing Vivino...")
    df = all_datasets['vivino']
    count = 0

    for _, row in tqdm(df.iterrows(), total=len(df)):
        name = row.get('Name', row.get('name', ''))
        if not name:
            continue

        wine_type = row.get('wine_type', '')
        winery = row.get('Winery', '')
        rating = row.get('Rating', '')
        price = row.get('Price', '')
        country = row.get('Country', '')
        region = row.get('Region', '')

        parts = []
        if winery:
            parts.append(f"{name} is produced by {winery}.")
        if wine_type:
            parts.append(f"It's a {wine_type.lower()} wine.")
        if region and country:
            parts.append(f"This wine comes from {region}, {country}.")
        if rating and str(rating) not in ['', 'nan', 'None']:
            parts.append(f"It has a Vivino rating of {rating}.")
        if price and str(price) not in ['', 'nan', 'None']:
            parts.append(f"It's priced around ${price}.")

        if parts:
            training_data.append(create_conversation(
                f"What can you tell me about {name}?", " ".join(parts)
            ))
            count += 1

    print(f"  {count:,} conversations")

Processing Vivino...


100%|██████████| 13834/13834 [00:00<00:00, 14783.52it/s]

  13,834 conversations


In [ ]:
if 'wikipedia' in all_datasets:
    print("Processing Wikipedia...")
    df = all_datasets['wikipedia']
    count = 0

    for _, row in df.iterrows():
        title = row.get('title', '')
        summary = row.get('summary', '')
        text = row.get('text', '')

        if not title or not summary:
            continue

        response = summary[:1500]
        if len(summary) > 1500:
            response = response.rsplit('.', 1)[0] + '.'

        training_data.append(create_conversation(f"What is {title}?", response))
        count += 1

        if text and len(text) > 500:
            detail = text[:2000]
            if len(text) > 2000:
                detail = detail.rsplit('.', 1)[0] + '.'

            training_data.append(create_conversation(
                f"Tell me more about {title}.", detail
            ))
            count += 1

    print(f"  {count:,} conversations")

Processing Wikipedia...
  92 conversations


In [ ]:
print(f"\nTotal before synthetic: {len(training_data):,} conversations")


Total before synthetic: 243,804 conversations


---

# Part 4: Synthetic Data Generation

Synthetic data is the secret sauce. Real wine reviews teach vocabulary, but they do not demonstrate *how a sommelier converses*. We use Gemini to generate realistic QA dialogues covering food pairings, wine knowledge, and recommendations.

In [ ]:
SYNTHETIC_TOPICS = {
    "food_pairing": [
        "grilled salmon", "beef steak", "spicy Thai curry", "mushroom risotto",
        "seared scallops", "barbecue ribs", "goat cheese salad", "lamb chops",
        "sushi", "Thanksgiving turkey", "pizza margherita", "butter chicken",
        "lobster", "vegetarian lasagna", "duck confit", "oysters",
        "blue cheese", "dark chocolate", "Mexican tacos", "pork tenderloin"
    ],
    "wine_knowledge": [
        "the difference between Chablis and other Chardonnays",
        "what malolactic fermentation does to wine",
        "why Burgundy wines are expensive",
        "how to tell if a wine is corked",
        "what minerality means in wine",
        "Old World vs New World wines",
        "why some wines need decanting",
        "how to read a French wine label",
        "what tannins are",
        "Champagne vs Prosecco",
        "what makes wine age-worthy",
        "why natural wines taste funky",
        "climate change and wine regions",
        "biodynamic winemaking",
        "proper wine serving temperatures"
    ],
    "recommendations": [
        "a bold red under $30",
        "a crisp white for summer",
        "a gift for a Burgundy lover",
        "a wine for someone who likes sweet wines",
        "a special occasion wine around $100",
        "an everyday wine under $15",
        "a smooth, fruity red",
        "a sparkling alternative to Champagne",
        "a wine to cellar for 10 years",
        "a crowd-pleaser for mixed tastes"
    ]
}

In [ ]:
import time

GEMINI_MODEL = "google/gemini-2.5-flash-lite"

def generate_synthetic(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = ai.generate_text(prompt, model_name=GEMINI_MODEL)
            return response
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return None

def parse_response(text):
    if not text or "USER:" not in text or "ASSISTANT:" not in text:
        return None

    try:
        parts = text.split("ASSISTANT:")
        user = parts[0].replace("USER:", "").strip()
        assistant = parts[1].strip()

        if len(user) > 10 and len(assistant) > 50:
            return create_conversation(user, assistant)
    except Exception:
        pass
    return None

In [ ]:
if GEMINI_AVAILABLE:
    synthetic_data = []

    print("Generating food pairing conversations...")
    for food in tqdm(SYNTHETIC_TOPICS["food_pairing"]):
        prompt = f"""You are Sommo, an expert sommelier. Generate a conversation about wine pairing for {food}.

Include: specific wine recommendations (grape + region), WHY they pair well, and a bottle suggestion with price.

Format:
USER: [question]
ASSISTANT: [response]"""

        result = parse_response(generate_synthetic(prompt))
        if result:
            synthetic_data.append(result)
        time.sleep(0.3)

    print("Generating wine knowledge conversations...")
    for topic in tqdm(SYNTHETIC_TOPICS["wine_knowledge"]):
        prompt = f"""You are Sommo, an expert sommelier. Generate a conversation about: {topic}

Give an accurate, engaging answer with specific examples.

Format:
USER: [question]
ASSISTANT: [response]"""

        result = parse_response(generate_synthetic(prompt))
        if result:
            synthetic_data.append(result)
        time.sleep(0.3)

    print("Generating recommendation conversations...")
    for req in tqdm(SYNTHETIC_TOPICS["recommendations"]):
        prompt = f"""You are Sommo, an expert sommelier. Someone asks for: {req}

Recommend 2-3 specific wines with producers and prices. Explain why each fits.

Format:
USER: [question]
ASSISTANT: [response]"""

        result = parse_response(generate_synthetic(prompt))
        if result:
            synthetic_data.append(result)
        time.sleep(0.3)

    training_data.extend(synthetic_data)
    print(f"\nGenerated {len(synthetic_data)} synthetic conversations")
else:
    print("Skipping synthetic generation (Gemini not available)")

Generating food pairing conversations...


100%|██████████| 20/20 [02:26<00:00,  7.35s/it]


Generating wine knowledge conversations...


100%|██████████| 15/15 [01:29<00:00,  5.98s/it]


Generating recommendation conversations...


100%|██████████| 10/10 [00:39<00:00,  3.93s/it]


Generated 44 synthetic conversations


---

# Part 5: Dataset Finalization

We shuffle the data and split 95/5 for training/validation.

In [ ]:
random.shuffle(training_data)

split = int(len(training_data) * 0.95)
train_data = training_data[:split]
val_data = training_data[split:]

print(f"Training: {len(train_data):,}")
print(f"Validation: {len(val_data):,}")

Training: 231,655
Validation: 12,193


In [ ]:
with open("data/processed/train.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

with open("data/processed/val.jsonl", "w") as f:
    for item in val_data:
        f.write(json.dumps(item) + "\n")

!cp data/processed/*.jsonl "{PROJECT_DIR}/data/"
print(f"Saved to {PROJECT_DIR}/data/")

Saved to /content/drive/MyDrive/sommo_ai/data/


In [ ]:
print("Sample conversation:\n")
sample = random.choice(training_data)
for msg in sample['conversations']:
    if msg['role'] != 'system':
        print(f"[{msg['role'].upper()}]")
        print(msg['content'][:500])
        print()

Sample conversation:

[USER]
Tell me about Greenwood Ridge 2013 Estate Bottled Merlot (Mendocino Ridge).

[ASSISTANT]
This Merlot from Mendocino Ridge, US is a noteworthy selection. A fruity and rich-tasting wine, this has black plum and mild oak aromas, ripe cherry and slightly savory, beefy flavors along with a nice, velvety texture. It is dry, medium bodied and drinker friendly. It earned 87 points from critics. At around $29, it represents solid value.



---

# Part 6: Model Training

We use [Unsloth](https://github.com/unslothai/unsloth) for efficient LoRA training.

| Parameter | Value | Why |
|-----------|-------|-----|
| LoRA rank | 64 | Good capacity without overfitting |
| Target modules | All attention + MLP | Comprehensive adaptation |
| Epochs | 3 | Optimal for domain adaptation |
| Batch size | 16 (effective) | Stable gradients |
| Learning rate | 2e-5 | Standard for LoRA |

Training takes approximately 3-4 hours on H100.

**Note:** You may see a warning about `num_items_in_batch` during training. This is expected behavior with Qwen models and gradient accumulation. The impact on accuracy is negligible. See [unsloth.ai/blog/gradient](https://unsloth.ai/blog/gradient) for details.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=False,
)

print(f"Loaded Qwen2.5-7B-Instruct ({model.num_parameters():,} parameters)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.2: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded Qwen2.5-7B-Instruct (7,615,616,512 parameters)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.1.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 161,480,704 || all params: 7,777,097,216 || trainable%: 2.0764


In [ ]:
from datasets import load_dataset

def format_to_chatml(examples):
    formatted = []
    for convs in examples['conversations']:
        text = ""
        for msg in convs:
            text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
        formatted.append(text)
    return {"text": formatted}

train_ds = load_dataset("json", data_files="data/processed/train.jsonl", split="train")
val_ds = load_dataset("json", data_files="data/processed/val.jsonl", split="train")

train_ds = train_ds.map(format_to_chatml, batched=True, remove_columns=train_ds.column_names)
val_ds = val_ds.map(format_to_chatml, batched=True, remove_columns=val_ds.column_names)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/231655 [00:00<?, ? examples/s]

Map:   0%|          | 0/12193 [00:00<?, ? examples/s]

Train: 231,655 | Val: 12,193


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/checkpoints",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    num_train_epochs=3,
    bf16=True,
    optim="adamw_8bit",
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=500,
    weight_decay=0.01,
    max_grad_norm=1.0,
    dataloader_num_workers=4,
    group_by_length=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
    max_seq_length=4096,
    dataset_text_field="text",
    packing=True,
)

print("Trainer ready")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/231655 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/12193 [00:00<?, ? examples/s]

Trainer ready


In [ ]:
print("Starting training...\n")
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting training...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 231,655 | Num Epochs = 3 | Total steps = 43,437
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 161,480,704 of 7,777,097,216 (2.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
500,0.971400,0.936520
1000,0.843400,0.834884
1500,0.818400,0.811412
2000,0.809400,0.795398
2500,0.770700,0.780541
3000,0.779400,0.768631
3500,0.761700,0.759420
4000,0.757100,0.750507
4500,0.749800,0.744424
5000,0.743000,0.739841


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=43437, training_loss=0.7218152962246469, metrics={'train_runtime': 82803.7273, 'train_samples_per_second': 8.393, 'train_steps_per_second': 0.525, 'total_flos': 6.262747464182953e+18, 'train_loss': 0.7218152962246469, 'epoch': 3.0})

In [ ]:
trainer.save_model(f"{PROJECT_DIR}/models/sommo-7b-v1-lora")
tokenizer.save_pretrained(f"{PROJECT_DIR}/models/sommo-7b-v1-lora")
print(f"Saved to {PROJECT_DIR}/models/sommo-7b-v1-lora")

Saved to /content/drive/MyDrive/sommo_ai/models/sommo-7b-v1-lora


---

# Part 7: Evaluation

Testing Sommo on representative questions.

In [ ]:
FastLanguageModel.for_inference(model)

def ask_sommo(question):
    prompt = f"<|im_start|>system\n{SOMMO_SYSTEM}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    return response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()

In [ ]:
TEST_QUESTIONS = [
    "What wine pairs best with grilled salmon?",
    "What's the difference between Champagne and Prosecco?",
    "I want a bold red wine under $30. Suggestions?",
    "Can you explain what tannins are?",
    "What's special about Burgundy wines?",
    "I only like sweet wines. How do I explore dry wines?",
]

for q in TEST_QUESTIONS:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"\nSommo: {ask_sommo(q)}")


Q: What wine pairs best with grilled salmon?

Sommo: Grilled salmon pairs wonderfully with '09 Cuvée de la Famille Pinot Noir (Willamette Valley). This is a light and easy-drinking wine that offers up flavors of cherries, cola and smoke. It has a fresh, clean finish. It earned 86 points from critics. At around $24, it represents solid value.

Q: What's the difference between Champagne and Prosecco?

Sommo: Champagne is produced from Pinot Meunier, Chardonnay and Pinot Noir grapes grown exclusively within the Champagne region of France, while Prosecco is made from the Glera (once known as Proseccu) grape in northeastern Italy. While both wines are sparkling, they differ greatly in flavor profiles.
Champagne tends to be fuller-bodied than Prosecco and shows aromas of fresh brioche, apple cider, white bread and yeast. Prosecco has a more delicate nose that includes pear blossoms, citrus zest and green apples. The palate on Champagne tends to be creamy and yeasty, whereas Prosecco boasts 

---

# Part 8: Export and Deployment

We merge the LoRA weights into the base model for easier deployment.

In [ ]:
from peft import PeftModel

print("Merging LoRA weights...")

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=4096,
)

merged_model = PeftModel.from_pretrained(base_model, f"{PROJECT_DIR}/models/sommo-7b-v1-lora")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(f"{PROJECT_DIR}/models/sommo-7b-v1")
base_tokenizer.save_pretrained(f"{PROJECT_DIR}/models/sommo-7b-v1")

print(f"Saved merged model to {PROJECT_DIR}/models/sommo-7b-v1")

Merging LoRA weights...
==((====))==  Unsloth 2026.1.2: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Saved merged model to /content/drive/MyDrive/sommo_ai/models/sommo-7b-v1


In [ ]:
PUSH_TO_HUB = True
HF_REPO = "gokhanarkan/sommo-7b-v1"

if PUSH_TO_HUB:
    from huggingface_hub import login

    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
    except Exception:
        login()

    merged_model.push_to_hub(HF_REPO, private=False)
    base_tokenizer.push_to_hub(HF_REPO, private=False)

    print(f"Pushed to https://huggingface.co/{HF_REPO}")

README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00002.safetensors:   1%|          | 46.3MB / 4.99GB            

  ...0002-of-00002.safetensors:   0%|          |  607kB / 2.16GB            

Saved model to https://huggingface.co/gokhanarkan/sommo-7b-v1


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpxjhgk72f/tokenizer.json:  73%|#######2  | 8.30MB / 11.4MB            

Pushed to https://huggingface.co/gokhanarkan/sommo-7b-v1


---

# Complete

**Sommo AI v1** is trained and ready for use.

## Quick Start

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("gokhanarkan/sommo-7b-v1")
tokenizer = AutoTokenizer.from_pretrained("gokhanarkan/sommo-7b-v1")
```

## Links

- **Model:** [huggingface.co/gokhanarkan/sommo-7b-v1](https://huggingface.co/gokhanarkan/sommo-7b-v1)
- **App:** [sommo.app](https://sommo.app) (uses v2)
- **Code:** This notebook

## Files (Google Drive)

```
sommo_ai/
  data/
    train.jsonl
    val.jsonl
  models/
    sommo-7b-v1-lora/
    sommo-7b-v1/
  checkpoints/
```

---

*This is v1 — a proof of concept. The [Sommo iOS app](https://sommo.app) uses an enhanced v2 model with additional training data.*